In [ ]:
import numpy as np
import pandas as pd
import paddle
from paddleocr import PaddleOCR
paddle.utils.run_check()             # should print success
ocr = PaddleOCR()                    # no errors now
print("✔ PaddleOCR ready")


In [ ]:
pipeline = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device="gpu"         # or "cpu"
    # ocr_version="PP-OCRv5"  # optional, v5 is default
)

path = 'test1.png'
outputs = pipeline.predict(path)

# iterate over each table found in the page
for i, res in enumerate(outputs):
    # your existing prints + saves
    print(f"\n=== Table {i} ===")
    res.print()
    res.save_to_img(f"./out/table_{i}.png")
    res.save_to_json(f"./out/table_{i}.json")

    # --- new: extract raw JSON ---
    j = res.json['res']
    texts = j["rec_texts"]   # list of strings
    boxes = np.array(j["rec_boxes"])  # shape (n,4): [x_min,y_min,x_max,y_max]

    # 1) compute box-centers and median height
    x_centers = (boxes[:,0] + boxes[:,2]) / 2
    y_centers = (boxes[:,1] + boxes[:,3]) / 2
    heights   = (boxes[:,3] - boxes[:,1])
    med_h = np.median(heights)

    # 2) sort all indices by y-center
    idx_sorted = np.argsort(y_centers)

    # 3) group into rows whenever vertical gap > half median height
    rows = []
    current = [idx_sorted[0]]
    for idx in idx_sorted[1:]:
        if abs(y_centers[idx] - y_centers[current[-1]]) < (med_h * 0.5):
            current.append(idx)
        else:
            rows.append(current)
            current = [idx]
    rows.append(current)

    # 4) within each row, sort by x-center and collect texts
    table = []
    for row in rows:
        row = sorted(row, key=lambda k: x_centers[k])
        table.append([texts[k] for k in row])

    # 5) display as DataFrame (uneven rows will be NaN-padded)
    df = pd.DataFrame(table)
    print(df.to_string(index=False, header=False))

